# Brain Tumor MRI — Trustworthy AI Revision Notebook (Track A: Metrics, Calibration & Multi-Seed)

**Owner:** Iqra Safdar
**Paper:** *Toward Trustworthy AI for Brain Tumor MRI Classification: A Multi-Pillar Clinical Readiness Framework*

This notebook rebuilds every metric the funders flagged as inconsistent or incomplete, and produces the
tables needed to replace Tables V–IX and Fig. 3–5 in the paper. It covers:

1. Confusion-matrix-derived accuracy/precision/recall/F1 (fixes the CM vs. Table VII mismatch)
2. Bootstrap 95% CIs recomputed per metric and explicitly labeled (accuracy CI ≠ recall CI ≠ F1 CI)
3. Five-seed training instead of three
4. HCE + Monte Carlo Dropout extended to ResNet50 and EfficientNetB0 (previously MobileNetV2-only)
5. NLL, Brier score, adaptive ECE, classwise ECE, and temperature-scaled ECE for all three architectures
6. Class distribution reporting + a check on scalar α=0.25 vs. class-balanced alternatives
7. Full CRI recomputed for all three architectures (Track B supplies the corrected `Gen` term)

Track B (Malaika) covers the Figshare leakage check, the CRI generalization/weighting writeup, and the
quantitative Grad-CAM audit — this notebook writes its outputs to `results/` in a format Track B's
notebook can read directly (`external_validation.json` for the `Gen` term).


## 0. Setup

In [1]:

import os, json, random, hashlib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from sklearn.metrics import confusion_matrix

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs available:", len(gpus))


TensorFlow: 2.20.0
GPUs available: 2


In [2]:

# ---------------------------------------------------------------------------
# CONFIG — point these at your environment. Defaults match the Kaggle
# "Brain Tumor MRI Dataset" layout (masoudnickparvar/brain-tumor-mri-dataset):
#   <root>/Training/{glioma,meningioma,notumor,pituitary}/*.jpg
#   <root>/Testing/{glioma,meningioma,notumor,pituitary}/*.jpg
# ---------------------------------------------------------------------------
CONFIG = {
    "data_root": "/kaggle/input/brain-tumor-mri-dataset",
    "class_names": ["glioma", "meningioma", "notumor", "pituitary"],
    "image_size": (224, 224),
    "batch_size": 32,
    "val_split": 0.20,
    "seeds": [42, 789, 999, 2023, 7],
    "architectures": ["mobilenetv2", "resnet50", "efficientnetb0"],
    "unfrozen_layers": 50,
    "dropout_rate": 0.5,
    "learning_rate": 1e-4,
    "max_epochs": 30,
    "early_stopping_patience": 10,
    "lr_reduce_factor": 0.5,
    "lr_reduce_patience": 4,
    "focal_gamma": 2.0,
    "focal_alpha": 0.25,
    "n_bootstrap": 10000,
    "hce_threshold": 0.9,
    "mc_dropout_passes": 10,
    "ece_bins": 10,
    "results_dir": "results",
    "models_dir": "models",
}
os.makedirs(CONFIG["results_dir"], exist_ok=True)
os.makedirs(CONFIG["models_dir"], exist_ok=True)

PREPROCESS_FN = {
    "mobilenetv2": mobilenet_preprocess,
    "resnet50": resnet_preprocess,
    "efficientnetb0": efficientnet_preprocess,
}
BASE_MODEL_FN = {
    "mobilenetv2": MobileNetV2,
    "resnet50": ResNet50,
    "efficientnetb0": EfficientNetB0,
}

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def find_dataset_root(base="/kaggle/input"):
    """Walk /kaggle/input looking for a folder that directly contains a
    train/test pair of subfolders, whatever they're capitalized as.
    Handles Kaggle mounting the dataset one level deeper than expected."""
    target_pairs = [("Training", "Testing"), ("Train", "Test"),
                     ("training", "testing"), ("train", "test")]
    for root, dirs, files in os.walk(base):
        dir_set = set(dirs)
        for train_name, test_name in target_pairs:
            if train_name in dir_set and test_name in dir_set:
                return root, train_name, test_name
    return None, None, None

_detected_root, _train_dir, _test_dir = find_dataset_root()
if _detected_root is not None:
    CONFIG["data_root"] = _detected_root
    CONFIG["train_dir_name"] = _train_dir
    CONFIG["test_dir_name"] = _test_dir
    print(f"Auto-detected dataset at: {_detected_root}  (train='{_train_dir}', test='{_test_dir}')")
else:
    CONFIG["train_dir_name"] = "Training"
    CONFIG["test_dir_name"] = "Testing"
    print("Could not auto-detect Training/Testing folders under /kaggle/input.")
    print("Full tree for manual inspection:")
    for root, dirs, files in os.walk("/kaggle/input"):
        print(root, "->", dirs)
    print('If you see the right folders above, set CONFIG["data_root"], '
          'CONFIG["train_dir_name"], CONFIG["test_dir_name"] manually in the next cell.')


Auto-detected dataset at: /kaggle/input/datasets/ekrasafdar/brain-tumor-mri  (train='Training', test='Testing')


## 1. Class distribution (funder point: report the [X] placeholders + justify scalar α)

Reads the actual per-class image counts from disk so Table III's training-set imbalance is reported
with real numbers instead of `[X]`, and computes what a class-balanced α vector would look like next to
the flat α=0.25 Lin et al. baseline actually used — this is the evidence for the α discussion in the text.

In [3]:

def get_class_distribution(root, split, class_names):
    counts = {}
    for c in class_names:
        d = os.path.join(root, split, c)
        counts[c] = len([f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))]) if os.path.isdir(d) else 0
    return counts

train_counts = get_class_distribution(CONFIG["data_root"], CONFIG["train_dir_name"], CONFIG["class_names"])
test_counts = get_class_distribution(CONFIG["data_root"], CONFIG["test_dir_name"], CONFIG["class_names"])

dist_df = pd.DataFrame({"train_count": train_counts, "test_count": test_counts})
dist_df["train_fraction"] = dist_df["train_count"] / dist_df["train_count"].sum()

total = dist_df["train_count"].sum()
n_classes = len(CONFIG["class_names"])
# Effective-number class weighting (Cui et al. 2019) as the class-balanced alternative to flat alpha=0.25
beta = 0.999
effective_num = 1.0 - np.power(beta, dist_df["train_count"].values)
class_balanced_weights = (1.0 - beta) / effective_num
class_balanced_weights = class_balanced_weights / class_balanced_weights.sum() * n_classes
dist_df["class_balanced_alpha"] = class_balanced_weights
dist_df["flat_alpha_used"] = CONFIG["focal_alpha"]

dist_df.to_csv(os.path.join(CONFIG["results_dir"], "class_distribution.csv"))
dist_df


,train_count,test_count,train_fraction,class_balanced_alpha,flat_alpha_used
glioma,1400,400,0.25,1.0,0.25
meningioma,1400,400,0.25,1.0,0.25
notumor,1400,400,0.25,1.0,0.25
pituitary,1400,400,0.25,1.0,0.25


## 2. Data pipelines (per-architecture preprocessing — keeps the EfficientNetB0 normalization bug from recurring)

In [4]:

def build_datasets(architecture, config, seed):
    preprocess_fn = PREPROCESS_FN[architecture]
    root = config["data_root"]

    train_ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join(root, config["train_dir_name"]),
        labels="inferred",
        label_mode="categorical",
        class_names=config["class_names"],
        image_size=config["image_size"],
        batch_size=config["batch_size"],
        validation_split=config["val_split"],
        subset="training",
        seed=seed,
        shuffle=True,
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join(root, config["train_dir_name"]),
        labels="inferred",
        label_mode="categorical",
        class_names=config["class_names"],
        image_size=config["image_size"],
        batch_size=config["batch_size"],
        validation_split=config["val_split"],
        subset="validation",
        seed=seed,
        shuffle=False,
    )
    test_ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join(root, config["test_dir_name"]),
        labels="inferred",
        label_mode="categorical",
        class_names=config["class_names"],
        image_size=config["image_size"],
        batch_size=config["batch_size"],
        shuffle=False,
    )

    augment = tf.keras.Sequential([
        layers.RandomRotation(20 / 360),
        layers.RandomTranslation(0.15, 0.15),
        layers.RandomZoom(0.15),
        layers.RandomFlip("horizontal"),
        layers.RandomBrightness(0.2),
    ])

    def prep_train(x, y):
        x = augment(x, training=True)
        x = preprocess_fn(x)
        return x, y

    def prep_eval(x, y):
        x = preprocess_fn(x)
        return x, y

    train_ds = train_ds.map(prep_train, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.map(prep_eval, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    test_ds = test_ds.map(prep_eval, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    return train_ds, val_ds, test_ds


## 3. Model definition

Outputs both a **logits** model and a **probability** model. The probability model is what trains and
what the paper's original head used; the logits model is required to do temperature scaling correctly
(you cannot temperature-scale an already-softmaxed output without re-deriving logits). `MCDropout` forces
dropout on at inference regardless of the outer `training` flag, without disturbing BatchNorm's inference
statistics — the standard fix for well-behaved Monte Carlo Dropout.

In [5]:

class MCDropout(layers.Dropout):
    def call(self, inputs, training=None):
        return super().call(inputs, training=True)

def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.math.pow(1 - y_pred, gamma)
        return tf.reduce_sum(weight * ce, axis=-1)
    return loss_fn

def build_model(architecture, config):
    input_shape = config["image_size"] + (3,)
    base = BASE_MODEL_FN[architecture](include_top=False, weights="imagenet", input_shape=input_shape)
    base.trainable = True
    for layer in base.layers[:-config["unfrozen_layers"]]:
        layer.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = MCDropout(config["dropout_rate"])(x)
    logits = layers.Dense(len(config["class_names"]), activation=None, name="logits")(x)
    probs = layers.Activation("softmax", name="probs")(logits)

    prob_model = models.Model(base.input, probs, name=f"{architecture}_probs")
    logit_model = models.Model(base.input, logits, name=f"{architecture}_logits")
    return prob_model, logit_model


## 4. Training loop — 5 seeds × 3 architectures

In [6]:

def train_one_run(architecture, seed, config):
    set_all_seeds(seed)
    train_ds, val_ds, test_ds = build_datasets(architecture, config, seed)
    prob_model, logit_model = build_model(architecture, config)

    prob_model.compile(
        optimizer=optimizers.Adam(config["learning_rate"]),
        loss=focal_loss(config["focal_gamma"], config["focal_alpha"]),
        metrics=["accuracy"],
    )

    cb = [
        callbacks.EarlyStopping(monitor="val_accuracy", patience=config["early_stopping_patience"],
                                 restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=config["lr_reduce_factor"],
                                     patience=config["lr_reduce_patience"]),
    ]

    prob_model.fit(train_ds, validation_data=val_ds, epochs=config["max_epochs"], callbacks=cb, verbose=2)

    weights_path = os.path.join(config["models_dir"], f"{architecture}_seed{seed}.weights.h5")
    prob_model.save_weights(weights_path)

    y_true, y_prob, logits = [], [], []
    for xb, yb in test_ds:
        p = prob_model.predict(xb, verbose=0)
        l = logit_model.predict(xb, verbose=0)
        y_true.append(np.argmax(yb.numpy(), axis=1))
        y_prob.append(p)
        logits.append(l)
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    logits = np.concatenate(logits)

    out_path = os.path.join(config["results_dir"], f"predictions_{architecture}_seed{seed}.npz")
    np.savez(out_path, y_true=y_true, y_prob=y_prob, logits=logits)
    return out_path

# Run everything. On a single GPU this is 15 training runs (5 seeds x 3 archs) —
# budget accordingly, or comment out architectures/seeds you've already run.
run_log = []
for architecture in CONFIG["architectures"]:
    for seed in CONFIG["seeds"]:
        print(f"=== Training {architecture} | seed {seed} ===")
        path = train_one_run(architecture, seed, CONFIG)
        run_log.append({"architecture": architecture, "seed": seed, "predictions_path": path})

pd.DataFrame(run_log).to_csv(os.path.join(CONFIG["results_dir"], "run_log.csv"), index=False)


=== Training mobilenetv2 | seed 42 ===
Found 5600 files belonging to 4 classes.
Using 4480 files for training.


I0000 00:00:1788616381.113700      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1788616381.116748      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 5600 files belonging to 4 classes.
Using 1120 files for validation.
Found 1600 files belonging to 4 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/30


2026-09-05 13:53:30.770694: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-05 13:53:30.907639: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1788616418.003461      78 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


140/140 - 67s - 481ms/step - accuracy: 0.7301 - loss: 0.1298 - val_accuracy: 0.2205 - val_loss: 0.4367 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 - 28s - 203ms/step - accuracy: 0.8478 - loss: 0.0637 - val_accuracy: 0.3821 - val_loss: 0.3871 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 - 29s - 207ms/step - accuracy: 0.8839 - loss: 0.0417 - val_accuracy: 0.6732 - val_loss: 0.1590 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 - 29s - 206ms/step - accuracy: 0.8958 - loss: 0.0339 - val_accuracy: 0.7759 - val_loss: 0.0964 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 - 28s - 203ms/step - accuracy: 0.9187 - loss: 0.0259 - val_accuracy: 0.8652 - val_loss: 0.0620 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 - 29s - 205ms/step - accuracy: 0.9270 - loss: 0.0212 - val_accuracy: 0.8839 - val_loss: 0.0500 - learning_rate: 1.0000e-04
Epoch 7/30
140/140 - 28s - 203ms/step - accuracy: 0.9368 - loss: 0.0171 - val_accuracy: 0.9563 - val_loss: 0.0141 - learning_rate: 1.0000e-04
Epoch 8/30
140/14

2026-09-05 16:07:44.729612: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-05 16:07:44.872209: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-05 16:07:45.222539: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-05 16:07:45.363346: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-05 16:07:46.164415: E external/local_xla/xla/stream_

140/140 - 68s - 489ms/step - accuracy: 0.6141 - loss: 0.1943 - val_accuracy: 0.7973 - val_loss: 0.0643 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 - 28s - 202ms/step - accuracy: 0.7699 - loss: 0.0923 - val_accuracy: 0.9384 - val_loss: 0.0187 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 - 28s - 202ms/step - accuracy: 0.8167 - loss: 0.0659 - val_accuracy: 0.9634 - val_loss: 0.0106 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 - 28s - 202ms/step - accuracy: 0.8471 - loss: 0.0530 - val_accuracy: 0.9705 - val_loss: 0.0081 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 - 28s - 200ms/step - accuracy: 0.8705 - loss: 0.0397 - val_accuracy: 0.9804 - val_loss: 0.0053 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 - 28s - 203ms/step - accuracy: 0.8846 - loss: 0.0361 - val_accuracy: 0.9777 - val_loss: 0.0053 - learning_rate: 1.0000e-04
Epoch 7/30
140/140 - 28s - 203ms/step - accuracy: 0.8978 - loss: 0.0298 - val_accuracy: 0.9848 - val_loss: 0.0037 - learning_rate: 1.0000e-04
Epoch 8/30
140/14

## 5. Confusion-matrix-derived metrics (fixes issue #1: CM vs. reported metrics disagree)

Every accuracy/precision/recall/F1 number reported from here on is computed **from the confusion matrix
directly** — never reported separately from a different code path — so Table VII-style tables can no
longer disagree with the matrix that accompanies them.

In [7]:

def metrics_from_confusion_matrix(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    accuracy = np.trace(cm) / cm.sum()

    per_class = {}
    for i, cls in enumerate(class_names):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        support = cm[i, :].sum()
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        per_class[cls] = {"precision": precision, "recall": recall, "f1": f1, "support": int(support)}

    weighted_f1 = sum(per_class[c]["f1"] * per_class[c]["support"] for c in class_names) / cm.sum()
    return cm, accuracy, per_class, weighted_f1

def load_predictions(architecture, seed, config):
    path = os.path.join(config["results_dir"], f"predictions_{architecture}_seed{seed}.npz")
    data = np.load(path)
    return data["y_true"], data["y_prob"], data["logits"]

metrics_rows = []
cm_store = {}
for architecture in CONFIG["architectures"]:
    for seed in CONFIG["seeds"]:
        y_true, y_prob, _ = load_predictions(architecture, seed, CONFIG)
        y_pred = np.argmax(y_prob, axis=1)
        cm, acc, per_class, w_f1 = metrics_from_confusion_matrix(y_true, y_pred, CONFIG["class_names"])
        cm_store[(architecture, seed)] = cm
        row = {"architecture": architecture, "seed": seed, "accuracy": acc, "weighted_f1": w_f1}
        for cls, m in per_class.items():
            row[f"{cls}_precision"] = m["precision"]
            row[f"{cls}_recall"] = m["recall"]
            row[f"{cls}_f1"] = m["f1"]
        metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(os.path.join(CONFIG["results_dir"], "per_seed_metrics_from_confusion_matrix.csv"), index=False)
metrics_df


,architecture,seed,accuracy,weighted_f1,glioma_precision,glioma_recall,glioma_f1,meningioma_precision,meningioma_recall,meningioma_f1,notumor_precision,notumor_recall,notumor_f1,pituitary_precision,pituitary_recall,pituitary_f1
0,mobilenetv2,42,0.896875,0.894796,0.942943,0.7850,0.856753,0.917847,0.8100,0.860558,0.927570,0.9925,0.958937,0.823045,1.0000,0.902935
1,mobilenetv2,789,0.924375,0.922965,0.953488,0.8200,0.881720,0.898219,0.8825,0.890290,0.952153,0.9950,0.973105,0.898876,1.0000,0.946746
2,mobilenetv2,999,0.922500,0.920905,0.993548,0.7700,0.867606,0.838202,0.9325,0.882840,0.949881,0.9950,0.971917,0.931925,0.9925,0.961259
3,mobilenetv2,2023,0.918750,0.917458,0.942529,0.8200,0.877005,0.890026,0.8700,0.879899,0.961071,0.9875,0.974106,0.886667,0.9975,0.938824
4,mobilenetv2,7,0.896875,0.895176,0.949555,0.8000,0.868385,0.882192,0.8050,0.841830,0.947368,0.9900,0.968215,0.827083,0.9925,0.902273
5,resnet50,42,0.891250,0.889071,0.993399,0.7525,0.856330,0.879679,0.8225,0.850129,0.917051,0.9950,0.954436,0.813906,0.9950,0.895388
6,resnet50,789,0.935000,0.933773,0.984802,0.8100,0.888889,0.873832,0.9350,0.903382,0.954545,0.9975,0.975550,0.938824,0.9975,0.967273
7,resnet50,999,0.899375,0.896827,0.993056,0.7150,0.831395,0.810384,0.8975,0.851720,0.961165,0.9900,0.975369,0.870897,0.9950,0.928821
8,resnet50,2023,0.872500,0.870384,0.972509,0.7075,0.819103,0.805825,0.8300,0.817734,0.955112,0.9575,0.956305,0.802419,0.9950,0.888393
9,resnet50,7,0.909375,0.907564,0.996753,0.7675,0.867232,0.894872,0.8725,0.883544,0.938967,1.0000,0.968523,0.838235,0.9975,0.910959


In [8]:

# Multi-seed summary (replaces Table V — now 5 seeds, mean +/- std, derived from the CM table above)
summary_cols = ["accuracy", "weighted_f1"]
summary = metrics_df.groupby("architecture")[summary_cols].agg(["mean", "std", "min", "max"])
summary.to_csv(os.path.join(CONFIG["results_dir"], "table_v_multiseed_summary.csv"))
summary


accuracy                               weighted_f1            \
                    mean       std       min       max        mean       std   
architecture                                                                   
efficientnetb0  0.901625  0.011764  0.887500  0.919375    0.899485  0.012178   
mobilenetv2     0.911875  0.013842  0.896875  0.924375    0.910260  0.014082   
resnet50        0.901500  0.023102  0.872500  0.935000    0.899524  0.023470   

                                    
                     min       max  
architecture                        
efficientnetb0  0.884912  0.917936  
mobilenetv2     0.894796  0.922965  
resnet50        0.870384  0.933773

## 6. Bootstrap confidence intervals — recomputed and explicitly labeled per metric (fixes issue #2)

Every CI below states which metric it belongs to. Resampling is done with replacement over the test set,
preserving the (y_true, y_pred) pairing per resample — the same protocol as the original paper, just
implemented so the labels can't drift from the numbers again.

In [9]:

def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=10000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    stats = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        stats[b] = metric_fn(y_true[idx], y_pred[idx])
    lower, upper = np.percentile(stats, [2.5, 97.5])
    return lower, upper, float(np.mean(stats))

def _accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def _class_recall(class_idx):
    def f(y_true, y_pred):
        mask = y_true == class_idx
        return np.mean(y_pred[mask] == class_idx) if mask.sum() > 0 else np.nan
    return f

def _class_precision(class_idx):
    def f(y_true, y_pred):
        mask = y_pred == class_idx
        return np.mean(y_true[mask] == class_idx) if mask.sum() > 0 else np.nan
    return f

def _class_f1(class_idx):
    prec_fn = _class_precision(class_idx)
    rec_fn = _class_recall(class_idx)
    def f(y_true, y_pred):
        p, r = prec_fn(y_true, y_pred), rec_fn(y_true, y_pred)
        return 0.0 if (p + r) == 0 or np.isnan(p) or np.isnan(r) else 2 * p * r / (p + r)
    return f

ci_rows = []
for architecture in CONFIG["architectures"]:
    for seed in CONFIG["seeds"]:
        y_true, y_prob, _ = load_predictions(architecture, seed, CONFIG)
        y_pred = np.argmax(y_prob, axis=1)

        lo, hi, mean_est = bootstrap_ci(y_true, y_pred, _accuracy, CONFIG["n_bootstrap"], seed)
        ci_rows.append({"architecture": architecture, "seed": seed, "metric": "accuracy",
                         "class": None, "point_estimate": np.mean(y_true == y_pred),
                         "ci_lower": lo, "ci_upper": hi})

        for c_idx, c_name in enumerate(CONFIG["class_names"]):
            for metric_name, fn_builder, fn in [
                ("recall", _class_recall, _class_recall(c_idx)),
                ("precision", _class_precision, _class_precision(c_idx)),
                ("f1", _class_f1, _class_f1(c_idx)),
            ]:
                lo, hi, mean_est = bootstrap_ci(y_true, y_pred, fn, CONFIG["n_bootstrap"], seed)
                point = fn(y_true, y_pred)
                ci_rows.append({"architecture": architecture, "seed": seed, "metric": metric_name,
                                 "class": c_name, "point_estimate": point,
                                 "ci_lower": lo, "ci_upper": hi})

ci_df = pd.DataFrame(ci_rows)
# Sanity check baked in: flag any interval that doesn't contain its own point estimate
ci_df["contains_point_estimate"] = (ci_df["point_estimate"] >= ci_df["ci_lower"]) & \
                                    (ci_df["point_estimate"] <= ci_df["ci_upper"])
assert ci_df["contains_point_estimate"].all(), "A CI does not contain its point estimate — recheck pairing."

ci_df.to_csv(os.path.join(CONFIG["results_dir"], "bootstrap_confidence_intervals.csv"), index=False)
ci_df.head(20)


,architecture,seed,metric,class,point_estimate,ci_lower,ci_upper,contains_point_estimate
0,mobilenetv2,42,accuracy,None,0.896875,0.882500,0.911250,True
1,mobilenetv2,42,recall,glioma,0.785000,0.744417,0.825871,True
2,mobilenetv2,42,precision,glioma,0.942943,0.917197,0.966464,True
3,mobilenetv2,42,f1,glioma,0.856753,0.828777,0.883291,True
4,mobilenetv2,42,recall,meningioma,0.810000,0.771357,0.847187,True
5,mobilenetv2,42,precision,meningioma,0.917847,0.887670,0.944928,True
6,mobilenetv2,42,f1,meningioma,0.860558,0.833111,0.886077,True
7,mobilenetv2,42,recall,notumor,0.992500,0.982885,1.000000,True
8,mobilenetv2,42,precision,notumor,0.927570,0.902552,0.951276,True
9,mobilenetv2,42,f1,notumor,0.958937,0.944942,0.972028,True


## 7. Calibration suite — ECE, adaptive ECE, classwise ECE, NLL, Brier score, temperature scaling
(fixes: "report NLL, Brier score, adaptive/classwise ECE, temperature-scaled results")

In [10]:

def compute_ece(y_true, y_prob, n_bins=10, adaptive=False):
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    correct = (predictions == y_true).astype(float)

    if adaptive:
        edges = np.unique(np.percentile(confidences, np.linspace(0, 100, n_bins + 1)))
    else:
        edges = np.linspace(0, 1, n_bins + 1)

    ece = 0.0
    n = len(confidences)
    for i in range(len(edges) - 1):
        lo, hi = edges[i], edges[i + 1]
        if i == len(edges) - 2:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() == 0:
            continue
        bin_acc = correct[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (mask.sum() / n) * abs(bin_acc - bin_conf)
    return ece

def compute_classwise_ece(y_true, y_prob, n_bins=10):
    n_classes = y_prob.shape[1]
    y_onehot = np.eye(n_classes)[y_true]
    per_class = []
    edges = np.linspace(0, 1, n_bins + 1)
    for c in range(n_classes):
        conf_c = y_prob[:, c]
        acc_c = y_onehot[:, c]
        ece_c = 0.0
        n = len(conf_c)
        for i in range(n_bins):
            lo, hi = edges[i], edges[i + 1]
            mask = (conf_c >= lo) & (conf_c <= hi) if i == n_bins - 1 else (conf_c >= lo) & (conf_c < hi)
            if mask.sum() == 0:
                continue
            ece_c += (mask.sum() / n) * abs(acc_c[mask].mean() - conf_c[mask].mean())
        per_class.append(ece_c)
    return float(np.mean(per_class)), per_class

def nll_loss(y_true, y_prob, eps=1e-12):
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return float(-np.mean(np.log(y_prob[np.arange(len(y_true)), y_true])))

def brier_score(y_true, y_prob):
    n_classes = y_prob.shape[1]
    y_onehot = np.eye(n_classes)[y_true]
    return float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))

def fit_temperature(logits, y_true, max_iter=200, lr=0.01):
    T = tf.Variable(1.0, dtype=tf.float32)
    logits_tf = tf.constant(logits, dtype=tf.float32)
    y_true_tf = tf.constant(y_true, dtype=tf.int32)
    opt = tf.optimizers.Adam(learning_rate=lr)
    for _ in range(max_iter):
        with tf.GradientTape() as tape:
            scaled_logits = logits_tf / T
            loss = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(y_true_tf, scaled_logits, from_logits=True)
            )
        grads = tape.gradient(loss, [T])
        opt.apply_gradients(zip(grads, [T]))
        T.assign(tf.maximum(T, 0.05))
    return float(T.numpy())

calibration_rows = []
for architecture in CONFIG["architectures"]:
    for seed in CONFIG["seeds"]:
        y_true, y_prob, logits = load_predictions(architecture, seed, CONFIG)

        ece = compute_ece(y_true, y_prob, CONFIG["ece_bins"], adaptive=False)
        adaptive_ece = compute_ece(y_true, y_prob, CONFIG["ece_bins"], adaptive=True)
        classwise_ece, classwise_per_class = compute_classwise_ece(y_true, y_prob, CONFIG["ece_bins"])
        nll = nll_loss(y_true, y_prob)
        brier = brier_score(y_true, y_prob)

        T = fit_temperature(logits, y_true)
        scaled_prob = tf.nn.softmax(logits / T, axis=1).numpy()
        ece_scaled = compute_ece(y_true, scaled_prob, CONFIG["ece_bins"], adaptive=False)
        nll_scaled = nll_loss(y_true, scaled_prob)

        calibration_rows.append({
            "architecture": architecture, "seed": seed,
            "ece": ece, "adaptive_ece": adaptive_ece, "classwise_ece": classwise_ece,
            "nll": nll, "brier_score": brier,
            "temperature": T, "ece_temp_scaled": ece_scaled, "nll_temp_scaled": nll_scaled,
        })

calibration_df = pd.DataFrame(calibration_rows)
calibration_df.to_csv(os.path.join(CONFIG["results_dir"], "calibration_suite_all_seeds.csv"), index=False)
calibration_df.groupby("architecture").mean(numeric_only=True)


,seed,ece,adaptive_ece,classwise_ece,nll,brier_score,temperature,ece_temp_scaled,nll_temp_scaled
architecture,,,,,,,,,
efficientnetb0,772.0,0.020760,0.017206,0.031348,0.355313,0.148467,1.460978,0.028292,0.321960
mobilenetv2,772.0,0.036742,0.036139,0.032997,0.399341,0.138498,1.846886,0.021539,0.303609
resnet50,772.0,0.040514,0.038098,0.038069,0.435526,0.152772,1.932787,0.024891,0.338170


## 8. High-Confidence Error rate (HCE) — now for all three architectures (fixes issue #6, part 1)

In [11]:

def compute_hce(y_true, y_prob, threshold=0.9):
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    incorrect = predictions != y_true
    if incorrect.sum() == 0:
        return 0.0
    return float(np.sum(incorrect & (confidences > threshold)) / incorrect.sum())

hce_rows = []
for architecture in CONFIG["architectures"]:
    for seed in CONFIG["seeds"]:
        y_true, y_prob, _ = load_predictions(architecture, seed, CONFIG)
        hce = compute_hce(y_true, y_prob, CONFIG["hce_threshold"])
        hce_rows.append({"architecture": architecture, "seed": seed, "hce": hce})

hce_df = pd.DataFrame(hce_rows)
hce_df.to_csv(os.path.join(CONFIG["results_dir"], "hce_all_architectures.csv"), index=False)
hce_df.groupby("architecture")["hce"].agg(["mean", "std"])


,mean,std
architecture,,
efficientnetb0,0.176680,0.014576
mobilenetv2,0.322555,0.068677
resnet50,0.293953,0.080498


## 9. Monte Carlo Dropout — now for all three architectures (fixes issue #6, part 2)

Rebuilds each architecture with `MCDropout` (already wired into `build_model`), reloads the saved weights
for one representative seed per architecture, and runs 10 stochastic forward passes to get predictive
entropy. Requires the test set to be rebuilt with that architecture's preprocessing.

In [12]:

def mc_dropout_predict(prob_model, x_batch, n_passes=10):
    preds = np.stack([prob_model(x_batch, training=False).numpy() for _ in range(n_passes)], axis=0)
    mean_probs = preds.mean(axis=0)
    entropy = -np.sum(mean_probs * np.log(np.clip(mean_probs, 1e-12, 1)), axis=1)
    return mean_probs, entropy

mc_dropout_rows = []
representative_seed = CONFIG["seeds"][0]  # seed 42, matching the original paper's reported instance
for architecture in CONFIG["architectures"]:
    prob_model, _ = build_model(architecture, CONFIG)
    weights_path = os.path.join(CONFIG["models_dir"], f"{architecture}_seed{representative_seed}.weights.h5")
    prob_model.load_weights(weights_path)

    _, _, test_ds = build_datasets(architecture, CONFIG, representative_seed)
    y_true_all, entropy_all, correct_all = [], [], []
    for xb, yb in test_ds:
        mean_probs, entropy = mc_dropout_predict(prob_model, xb, CONFIG["mc_dropout_passes"])
        y_true_batch = np.argmax(yb.numpy(), axis=1)
        pred_batch = np.argmax(mean_probs, axis=1)
        y_true_all.append(y_true_batch)
        entropy_all.append(entropy)
        correct_all.append(pred_batch == y_true_batch)

    y_true_all = np.concatenate(y_true_all)
    entropy_all = np.concatenate(entropy_all)
    correct_all = np.concatenate(correct_all)

    per_sample = pd.DataFrame({
        "architecture": architecture, "y_true": y_true_all,
        "entropy": entropy_all, "correct": correct_all,
    })
    mc_dropout_rows.append(per_sample)

mc_dropout_df = pd.concat(mc_dropout_rows, ignore_index=True)
mc_dropout_df.to_csv(os.path.join(CONFIG["results_dir"], "mc_dropout_entropy_all_architectures.csv"), index=False)
mc_dropout_df.groupby(["architecture", "correct"])["entropy"].describe()


Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 5600 files belonging to 4 classes.
Using 1120 files for validation.
Found 1600 files belonging to 4 classes.
Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 5600 files belonging to 4 classes.
Using 1120 files for validation.
Found 1600 files belonging to 4 classes.
Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 5600 files belonging to 4 classes.
Using 1120 files for validation.
Found 1600 files belonging to 4 classes.


count      mean       std           min       25%  \
architecture   correct                                                       
efficientnetb0 False     125.0  0.750574  0.313044  2.385638e-03  0.581292   
               True     1475.0  0.208690  0.276833  1.041556e-13  0.007127   
mobilenetv2    False     162.0  0.617081  0.310382  1.209128e-05  0.390589   
               True     1438.0  0.116454  0.225145  8.180010e-15  0.000065   
resnet50       False     161.0  0.663667  0.323952  4.652152e-03  0.442704   
               True     1439.0  0.137530  0.247373  8.445124e-18  0.000154   

                             50%       75%       max  
architecture   correct                                
efficientnetb0 False    0.774338  0.985757  1.335449  
               True     0.067693  0.328922  1.168221  
mobilenetv2    False    0.666144  0.814743  1.253168  
               True     0.002409  0.093629  1.102369  
resnet50       False    0.701841  0.949434  1.302692  
               True     0.006781  0.149453  1.351081

## 10. Clinical Readiness Index — recomputed for all three architectures (fixes issue #6, part 3)

`Gen` is read from `results/external_validation.json`, which Track B's notebook writes (the corrected
generalization ratio, post-leakage-check). Until that file exists, `Gen` falls back to `None` and is
reported as missing rather than silently defaulting to 1.0 — the original bug.

In [13]:

def load_external_validation(config):
    path = os.path.join(config["results_dir"], "external_validation.json")
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)  # expected: {"architecture": {"external_accuracy": .., "gen_ratio": ..}}
    return {}

def compute_cri(acc, ece, hce, gen, weights=(0.40, 0.25, 0.20, 0.15)):
    w_acc, w_cal, w_safety, w_gen = weights
    if gen is None:
        return None
    return w_acc * acc + w_cal * (1 - ece) + w_safety * (1 - hce) + w_gen * gen

external = load_external_validation(CONFIG)

cri_rows = []
mean_acc = metrics_df.groupby("architecture")["accuracy"].mean()
mean_ece = calibration_df.groupby("architecture")["ece"].mean()
mean_hce = hce_df.groupby("architecture")["hce"].mean()

for architecture in CONFIG["architectures"]:
    gen = external.get(architecture, {}).get("gen_ratio")
    cri = compute_cri(mean_acc[architecture], mean_ece[architecture], mean_hce[architecture], gen)
    cri_rows.append({
        "architecture": architecture,
        "acc_term": 0.40 * mean_acc[architecture],
        "cal_term": 0.25 * (1 - mean_ece[architecture]),
        "safety_term": 0.20 * (1 - mean_hce[architecture]),
        "gen_term": 0.15 * gen if gen is not None else None,
        "cri": cri,
        "gen_source": "results/external_validation.json (Track B)" if gen is not None else "MISSING — awaiting Track B",
    })

cri_df = pd.DataFrame(cri_rows)
cri_df.to_csv(os.path.join(CONFIG["results_dir"], "cri_all_architectures.csv"), index=False)
cri_df


,architecture,acc_term,cal_term,safety_term,gen_term,cri,gen_source
0,mobilenetv2,0.36475,0.240815,0.135489,None,None,MISSING — awaiting Track B
1,resnet50,0.36060,0.239872,0.141209,None,None,MISSING — awaiting Track B
2,efficientnetb0,0.36065,0.244810,0.164664,None,None,MISSING — awaiting Track B


In [14]:

# CRI sensitivity to weight choice, for every architecture (extends the original MobileNetV2-only table)
weight_schemes = {
    "baseline (0.40/0.25/0.20/0.15)": (0.40, 0.25, 0.20, 0.15),
    "equal (0.25/0.25/0.25/0.25)": (0.25, 0.25, 0.25, 0.25),
    "safety-heavy (0.25/0.25/0.35/0.15)": (0.25, 0.25, 0.35, 0.15),
    "accuracy-heavy (0.55/0.20/0.15/0.10)": (0.55, 0.20, 0.15, 0.10),
}

sensitivity_rows = []
for architecture in CONFIG["architectures"]:
    gen = external.get(architecture, {}).get("gen_ratio")
    for scheme_name, weights in weight_schemes.items():
        cri = compute_cri(mean_acc[architecture], mean_ece[architecture], mean_hce[architecture], gen, weights)
        sensitivity_rows.append({"architecture": architecture, "weight_scheme": scheme_name, "cri": cri})

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df.to_csv(os.path.join(CONFIG["results_dir"], "cri_weight_sensitivity_all_architectures.csv"), index=False)
sensitivity_df.pivot(index="architecture", columns="weight_scheme", values="cri")


weight_scheme,accuracy-heavy (0.55/0.20/0.15/0.10),baseline (0.40/0.25/0.20/0.15),equal (0.25/0.25/0.25/0.25),safety-heavy (0.25/0.25/0.35/0.15)
architecture,,,,
efficientnetb0,None,None,None,None
mobilenetv2,None,None,None,None
resnet50,None,None,None,None


## 11. Output manifest

Everything below lands in `results/` and is what Malaika's notebook (Track B) and the paper's revised
tables should read from — nothing here should need hand-editing before it goes into a table.

In [15]:

manifest = sorted(os.listdir(CONFIG["results_dir"]))
for f in manifest:
    print(f)


bootstrap_confidence_intervals.csv
calibration_suite_all_seeds.csv
class_distribution.csv
cri_all_architectures.csv
cri_weight_sensitivity_all_architectures.csv
hce_all_architectures.csv
mc_dropout_entropy_all_architectures.csv
per_seed_metrics_from_confusion_matrix.csv
predictions_efficientnetb0_seed2023.npz
predictions_efficientnetb0_seed42.npz
predictions_efficientnetb0_seed7.npz
predictions_efficientnetb0_seed789.npz
predictions_efficientnetb0_seed999.npz
predictions_mobilenetv2_seed2023.npz
predictions_mobilenetv2_seed42.npz
predictions_mobilenetv2_seed7.npz
predictions_mobilenetv2_seed789.npz
predictions_mobilenetv2_seed999.npz
predictions_resnet50_seed2023.npz
predictions_resnet50_seed42.npz
predictions_resnet50_seed7.npz
predictions_resnet50_seed789.npz
predictions_resnet50_seed999.npz
run_log.csv
table_v_multiseed_summary.csv
